# 08. Analyse du Dataset et Amélioration des Scores

**Objectif** : Analyser en détail le dataset pour identifier les améliorations possibles

**Pour les data scientists** : Ce notebook examine les données et propose des améliorations

In [ ]:
# Importer les bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
import re
import time

warnings.filterwarnings('ignore')

# Configuration
%matplotlib inline
try:
    plt.style.use('seaborn-v0_8')
except:
    try:
        plt.style.use('seaborn')
    except:
        plt.style.use('default')

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("📚 Bibliothèques importées avec succès !")
print("🔍 Analyse du Dataset et Amélioration des Scores")

In [ ]:
# Charger et analyser le dataset original
print("=== CHARGEMENT DU DATASET ORIGINAL ===")

# Charger les données
df = pd.read_csv("../data/real_estate_processed.csv")
print(f"Dataset original: {df.shape}")
print(f"Colonnes: {list(df.columns)}")

# Afficher les premières lignes
print("\n📋 Aperçu des données:")
display(df.head())

# Informations générales
print(f"\n📊 INFORMATIONS GÉNÉRALES:")
print(f"   Lignes: {len(df):,}")
print(f"   Colonnes: {len(df.columns)}")
print(f"   Valeurs manquantes: {df.isnull().sum().sum()}")
print(f"   Types de données: {df.dtypes.value_counts().to_dict()}")

# Statistiques des prix
print(f"\n💰 STATISTIQUES DES PRIX:")
print(f"   Moyenne: {df['price'].mean():,.0f} DT")
print(f"   Médiane: {df['price'].median():,.0f} DT")
print(f"   Écart-type: {df['price'].std():,.0f} DT")
print(f"   Min: {df['price'].min():,.0f} DT")
print(f"   Max: {df['price'].max():,.0f} DT")
print(f"   Skewness: {stats.skew(df['price']):.4f}")
print(f"   Kurtosis: {stats.kurtosis(df['price']):.4f}")

In [ ]:
# Implémentation des améliorations
print("=== IMPLÉMENTATION DES AMÉLIORATIONS ===")

# Créer une copie du dataset pour les améliorations
df_improved = df.copy()

print(f"Dataset original: {df.shape}")

# 1. Nettoyage des données
print("\n🧹 1. NETTOYAGE DES DONNÉES")

# Supprimer les doublons
duplicates_before = len(df_improved)
df_improved = df_improved.drop_duplicates()
duplicates_after = len(df_improved)
print(f"   Doublons supprimés: {duplicates_before - duplicates_after}")

# Gérer les valeurs manquantes - éviter le problème avec les colonnes catégorielles
missing_before = df_improved.isnull().sum().sum()

# Identifier les colonnes catégorielles
categorical_cols = df_improved.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = df_improved.select_dtypes(include=[np.number]).columns.tolist()

print(f"   Colonnes catégorielles: {categorical_cols}")
print(f"   Colonnes numériques: {numeric_cols}")

# Remplir les valeurs manquantes par colonne
for col in numeric_cols:
    df_improved[col] = df_improved[col].fillna(df_improved[col].median())

for col in categorical_cols:
    df_improved[col] = df_improved[col].fillna('Unknown')

missing_after = df_improved.isnull().sum().sum()
print(f"   Valeurs manquantes traitées: {missing_before}")

# 2. Conversion de la colonne 'rooms' en numérique
print("\n🔢 2. CONVERSION DE LA COLONNE 'ROOMS'")

# Fonction pour extraire le nombre de pièces
def extract_rooms(room_str):
    if pd.isna(room_str):
        return np.nan
    if isinstance(room_str, (int, float)):
        return int(room_str)
    
    room_str = str(room_str).strip()
    
    # Cas spécial pour 'S+X' format
    if room_str.startswith('S+'):
        try:
            return int(room_str[2:])
        except:
            return np.nan
    
    # Cas pour les nombres simples
    try:
        return int(room_str)
    except:
        return np.nan

# Appliquer la conversion
df_improved['rooms_numeric'] = df_improved['rooms'].apply(extract_rooms)
df_improved['rooms_numeric'] = df_improved['rooms_numeric'].fillna(df_improved['rooms_numeric'].median())
df_improved['rooms'] = df_improved['rooms_numeric']  # Remplacer la colonne originale

print(f"   Colonne 'rooms' convertie en numérique")
print(f"   Valeurs uniques après conversion: {sorted(df_improved['rooms'].unique())[:10]}...")

# 3. Features textuelles
print("\n📝 3. FEATURES TEXTUELLES")

# Extraire la surface
def extract_surface(title):
    if pd.isna(title):
        return np.nan
    patterns = [r'(\d+)\s*m[²2]', r'(\d+)\s*m²', r'(\d+)\s*metre[²2]', r'(\d+)\s*metres[²2]']
    for pattern in patterns:
        match = re.search(pattern, str(title).lower())
        if match:
            try:
                return int(match.group(1))
            except:
                return np.nan
    return np.nan

df_improved['surface'] = df_improved['title'].apply(extract_surface)
df_improved['surface'] = df_improved['surface'].fillna(df_improved['surface'].median())
print(f"   Surface extraite pour {df_improved['surface'].notna().sum()} annonces")

# Features binaires
df_improved['is_meublé'] = df_improved['title'].str.contains('meublé|meublée|fourni|équipé', case=False, na=False).astype(int)
df_improved['has_clim'] = df_improved['title'].str.contains('clim|climatisation|air|chauffage|cv', case=False, na=False).astype(int)
df_improved['has_parking'] = df_improved['title'].str.contains('parking|garage|box|stationnement', case=False, na=False).astype(int)
df_improved['vue_mer'] = df_improved['title'].str.contains('vue.*mer|mer.*vue|front.*mer|bord.*mer', case=False, na=False).astype(int)
df_improved['has_jardin'] = df_improved['title'].str.contains('jardin|cour|terrasse|balcon', case=False, na=False).astype(int)
df_improved['is_haut_standing'] = df_improved['title'].str.contains('haut.*standing|standing.*haut|prestige|luxe|premium', case=False, na=False).astype(int)
df_improved['is_neuf'] = df_improved['title'].str.contains('neuf|neuve|recent|construction.*neuve', case=False, na=False).astype(int)

print(f"   Features binaires créées: {['is_meublé', 'has_clim', 'has_parking', 'vue_mer', 'has_jardin', 'is_haut_standing', 'is_neuf']}")

# 4. Features temporelles avancées
print("\n📅 4. FEATURES TEMPORELLES AVANCÉES")

# Convertir les dates si nécessaire
df_improved['post_date'] = pd.to_datetime(df_improved['post_date'], format='%m-%d-%y', errors='coerce')
df_improved['post_time'] = pd.to_datetime(df_improved['post_time'], format='%m/%d/%y %H:%M', errors='coerce')

# Features temporelles
df_improved['post_day'] = df_improved['post_date'].dt.day
df_improved['post_dayofweek'] = df_improved['post_date'].dt.dayofweek
df_improved['post_weekend'] = (df_improved['post_dayofweek'] >= 5).astype(int)
df_improved['post_hour'] = df_improved['post_time'].dt.hour
df_improved['season'] = df_improved['post_month'].map({
    12: 'Hiver', 1: 'Hiver', 2: 'Hiver',
    3: 'Printemps', 4: 'Printemps', 5: 'Printemps',
    6: 'Été', 7: 'Été', 8: 'Été',
    9: 'Automne', 10: 'Automne', 11: 'Automne'
})

# Ancienneté de l'annonce
current_date = pd.Timestamp.now()
df_improved['annonce_age_days'] = (current_date - df_improved['post_date']).dt.days
df_improved['annonce_age_weeks'] = df_improved['annonce_age_days'] // 7

print(f"   Features temporelles ajoutées: post_day, post_dayofweek, post_weekend, post_hour, season, annonce_age_days")

# 5. Features d'interaction
print("\n🔄 5. FEATURES D'INTERACTION")

# Interactions
df_improved['category_location'] = df_improved['category'].astype(str) + '_' + df_improved['location'].astype(str)
df_improved['price_per_surface'] = df_improved['price'] / df_improved['surface']
df_improved['price_per_room'] = df_improved['price'] / (df_improved['rooms'] + 1)
df_improved['rooms_per_surface'] = df_improved['rooms'] / df_improved['surface']
df_improved['category_rooms'] = df_improved['category'] * df_improved['rooms']

print(f"   Features d'interaction créées: category_location, price_per_surface, price_per_room, rooms_per_surface, category_rooms")

# 6. Features polynomiales
print("\n📈 6. FEATURES POLYNOMIALES")

# Features polynomiales
df_improved['price_squared'] = df_improved['price'] ** 2
df_improved['rooms_squared'] = df_improved['rooms'] ** 2
df_improved['category_squared'] = df_improved['category'] ** 2
df_improved['surface_squared'] = df_improved['surface'] ** 2

print(f"   Features polynomiales créées: price_squared, rooms_squared, category_squared, surface_squared")

# 7. Features de localisation améliorées
print("\n📍 7. FEATURES DE LOCALISATION AMÉLIORÉES")

# Target encoding pour localisation
location_stats = df_improved.groupby('location')['price'].agg(['mean', 'median', 'std', 'count'])
df_improved['location_price_mean'] = df_improved['location'].map(location_stats['mean'])
df_improved['location_price_median'] = df_improved['location'].map(location_stats['median'])
df_improved['location_price_std'] = df_improved['location'].map(location_stats['std'])

print(f"   Features de localisation: location_price_mean, location_price_median, location_price_std")

# 8. Features riches
print("\n💎 8. FEATURES RICHES")

# Features logarithmiques
df_improved['log_price'] = np.log1p(df_improved['price'])
df_improved['log_surface'] = np.log1p(df_improved['surface'])
df_improved['log_rooms'] = np.log1p(df_improved['rooms'])

# Ratios
df_improved['surface_per_room'] = df_improved['surface'] / (df_improved['rooms'] + 1)
df_improved['price_category_ratio'] = df_improved['price'] / (df_improved['category'] + 1)

print(f"   Features riches: log_price, log_surface, log_rooms, surface_per_room, price_category_ratio")

# Afficher les résultats
print(f"\n📊 RÉSULTATS DES AMÉLIORATIONS:")
print(f"   Dataset original: {df.shape}")
print(f"   Dataset amélioré: {df_improved.shape}")
print(f"   Nouvelles features: {df_improved.shape[1] - df.shape[1]}")

# Afficher les nouvelles colonnes
new_columns = [col for col in df_improved.columns if col not in df.columns]
print(f"\n🔧 NOUVELLES COLONNES ({len(new_columns)}):")
for i, col in enumerate(new_columns, 1):
    print(f"   {i:2d}. {col}")

# Statistiques des nouvelles features
print(f"\n📈 STATISTIQUES DES NOUVELLES FEATURES:")
for col in new_columns[:10]:  # Limiter aux 10 premières
    if df_improved[col].dtype in ['int64', 'float64']:
        print(f"   {col:25s}: min={df_improved[col].min():.2f}, max={df_improved[col].max():.2f}, mean={df_improved[col].mean():.2f}")
    else:
        print(f"   {col:25s}: {df_improved[col].nunique()} valeurs uniques")

In [ ]:
# Test des améliorations avec un modèle simple
print("=== TEST DES AMÉLIORATIONS AVEC UN MODÈLE SIMPLE ===")

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
import numpy as np

print("\n📊 TEST AVEC DONNÉES ORIGINALES:")

# Préparer les données originales
df_orig = df.copy()

# Conversion de la colonne 'rooms' pour les données originales
def extract_rooms(room_str):
    if pd.isna(room_str):
        return np.nan
    if isinstance(room_str, (int, float)):
        return int(room_str)
    
    room_str = str(room_str).strip()
    
    # Cas spécial pour 'S+X' format
    if room_str.startswith('S+'):
        try:
            return int(room_str[2:])
        except:
            return np.nan
    
    # Cas pour les nombres simples
    try:
        return int(room_str)
    except:
        return np.nan

df_orig['rooms'] = df_orig['rooms'].apply(extract_rooms)
df_orig['rooms'] = df_orig['rooms'].fillna(df_orig['rooms'].median())

# Sélectionner les features originaux
original_features = ['category', 'type_transaction', 'rooms', 'post_month', 'post_year']
X_orig = df_orig[original_features].copy()
y_orig = df_orig['price']

# Encoder les variables catégorielles
categorical_cols = ['category', 'type_transaction']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat_encoded = encoder.fit_transform(X_orig[categorical_cols])

# Combiner les features
X_numeric = X_orig[['rooms', 'post_month', 'post_year']].values
X_orig_final = np.hstack([X_numeric, X_cat_encoded])

# Diviser les données
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_orig_final, y_orig, test_size=0.2, random_state=42
)

# Entraîner le modèle original
rf_orig = RandomForestRegressor(n_estimators=100, random_state=42)
rf_orig.fit(X_train_orig, y_train_orig)

# Prédire et évaluer
y_pred_orig = rf_orig.predict(X_test_orig)

# Calculer les métriques
r2_orig = r2_score(y_test_orig, y_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)

print(f"   R²: {r2_orig:.4f}")
print(f"   RMSE: {rmse_orig:,.0f} DT")
print(f"   MAE: {mae_orig:,.0f} DT")

print("\n📊 TEST AVEC DONNÉES AMÉLIORÉES:")

# Préparer les données améliorées
# Sélectionner uniquement les colonnes numériques pour le modèle
numeric_features = df_improved.select_dtypes(include=[np.number]).columns.tolist()
# Exclure les colonnes non pertinentes ou cibles
exclude_cols = ['price', 'log_price', 'price_squared', 'location_price_mean', 'location_price_median', 'location_price_std']
improved_features = [col for col in numeric_features if col not in exclude_cols]

X_imp = df_improved[improved_features].copy()
y_imp = df_improved['price']

# Diviser les données
X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_imp, y_imp, test_size=0.2, random_state=42
)

# Entraîner le modèle amélioré
rf_imp = RandomForestRegressor(n_estimators=100, random_state=42)
rf_imp.fit(X_train_imp, y_train_imp)

# Prédire et évaluer
y_pred_imp = rf_imp.predict(X_test_imp)

# Calculer les métriques
r2_imp = r2_score(y_test_imp, y_pred_imp)
rmse_imp = np.sqrt(mean_squared_error(y_test_imp, y_pred_imp))
mae_imp = mean_absolute_error(y_test_imp, y_pred_imp)

print(f"   R²: {r2_imp:.4f}")
print(f"   RMSE: {rmse_imp:,.0f} DT")
print(f"   MAE: {mae_imp:,.0f} DT")

# Calculer l'amélioration
improvement_r2 = r2_imp - r2_orig
rmse_improvement = (rmse_orig - rmse_imp) / rmse_orig * 100
mae_improvement = (mae_orig - mae_imp) / mae_orig * 100

print(f"\n🎯 AMÉLIORATION:")
print(f"   R²: {improvement_r2:+.4f}")
print(f"   RMSE: {rmse_improvement:+.2f}%")
print(f"   MAE: {mae_improvement:+.2f}%")

# Feature importance pour le modèle amélioré
feature_importance = pd.DataFrame({
    'feature': improved_features,
    'importance': rf_imp.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🔝 TOP 10 FEATURES IMPORTANTES:")
for i, (feat, imp) in enumerate(zip(feature_importance['feature'][:10], feature_importance['importance'][:10]), 1):
    print(f"   {i:2d}. {feat:30s}: {imp:.4f}")

print(f"\n✅ Tests terminés avec succès!")

In [ ]:
# Sauvegarder le dataset amélioré
print("=== SAUVEGARDE DU DATASET AMÉLIORÉ ===")

df_improved.to_csv("../data/real_estate_improved.csv", index=False)
print("✅ Dataset amélioré sauvegardé: ../data/real_estate_improved.csv")

# Sauvegarder les résultats
results = {
    'original_performance': {
        'r2': r2_orig,
        'rmse': rmse_orig,
        'mae': mae_orig,
        'features': original_features
    },
    'improved_performance': {
        'r2': r2_imp,
        'rmse': rmse_imp,
        'mae': mae_imp,
        'features': improved_features
    },
    'improvement': {
        'r2_improvement': improvement_r2,
        'rmse_improvement': (rmse_orig - rmse_imp) / rmse_orig * 100,
        'mae_improvement': (mae_orig - mae_imp) / mae_orig * 100
    },
    'new_features': new_columns,
    'feature_importance': feature_importance.to_dict('records')
}

import joblib
joblib.dump(results, '../models/dataset_improvement_results.pkl')

print(f"Dataset amélioré: {df_improved.shape}")
print(f"🔧 {len(new_columns)} nouvelles features créées")
print(f"💾 Tous les résultats sauvegardés")

In [ ]:
# Vérification finale
print("=== VÉRIFICATION FINALE ===")

# Vérifier que toutes les variables nécessaires existent
required_vars = ['r2_orig', 'rmse_orig', 'mae_orig', 'r2_imp', 'rmse_imp', 'mae_imp', 
                 'improvement_r2', 'original_features', 'improved_features', 
                 'new_columns', 'feature_importance']

missing_vars = []
for var in required_vars:
    if var not in locals():
        missing_vars.append(var)

if missing_vars:
    print(f"❌ Variables manquantes: {missing_vars}")
else:
    print("✅ Toutes les variables requises sont définies")

# Vérifier les types de données
print(f"\n📊 VÉRIFICATION DES TYPES DE DONNÉES:")
print(f"   df shape: {df.shape}")
print(f"   df_improved shape: {df_improved.shape}")
print(f"   Colonnes ajoutées: {len(new_columns)}")

# Vérifier que la colonne 'rooms' est bien numérique
if df_improved['rooms'].dtype in ['int64', 'float64']:
    print(f"✅ Colonne 'rooms' est numérique: {df_improved['rooms'].dtype}")
else:
    print(f"❌ Colonne 'rooms' n'est pas numérique: {df_improved['rooms'].dtype}")

# Vérifier qu'il n'y a pas de valeurs manquantes
missing_count = df_improved.isnull().sum().sum()
if missing_count == 0:
    print(f"✅ Aucune valeur manquante dans le dataset amélioré")
else:
    print(f"❌ {missing_count} valeurs manquantes trouvées")

# Résumé des performances
print(f"\n📈 RÉSUMÉ DES PERFORMANCES:")
print(f"   Modèle original - R²: {r2_orig:.4f}, RMSE: {rmse_orig:,.0f} DT, MAE: {mae_orig:,.0f} DT")
print(f"   Modèle amélioré - R²: {r2_imp:.4f}, RMSE: {rmse_imp:,.0f} DT, MAE: {mae_imp:,.0f} DT")
print(f"   Amélioration R²: {improvement_r2:+.4f}")

print(f"\n🎉 NOTEBOOK CORRIGÉ AVEC SUCCÈS!")